©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、NumPyのみを使ってシンプルな全結合ニューラルネットワーク（FCNN）をスクラッチ実装します。数値微分による勾配計算、交差エントロピー誤差、Softmax関数を自作し、順伝播から逆伝播・パラメータ更新までの一連の学習フローを基礎から確認します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）


In [ ]:
# %%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

# !pip uninstall torch -y
# !pip install torch==2.7.0

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0

# 学習まとめ

シンプルなネットワークを動かして一連の動きを確認する。

In [ ]:
# ライブラリのインポート
import numpy as np

In [ ]:
# 勾配計算関数の定義
def numerical_gradient(f, x):
    h = 1e-4                  # 微小な値hの設定
    grad = np.zeros_like(x)   # 勾配を格納するための配列を初期化

    # イテレータで要素に参照する場合の要素への読み書きモードを指定
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index      # 現在のインデックスを取得
        tmp_val = x[idx]          # 元の値を一時的に保存
        x[idx] = float(tmp_val) + h   # x+hの計算
        fxh1 = f(x)               # f(x+h)の計算

        x[idx] = tmp_val - h      # x-hの計算
        fxh2 = f(x)               # f(x-h)の計算
        grad[idx] = (fxh1 - fxh2) / (2*h)   # 勾配の計算

        x[idx] = tmp_val          # 値を元に戻す
        it.iternext()             # イテレータを次に進める

    return grad   # 勾配を返す

In [ ]:
# 交差エントロピー誤差
def cross_entropy_error(y, t):
    return np.sum(-t*np.log(y)-(1-t)*np.log(1-y))

In [ ]:
# Softmax関数
def softmax(x):
    exp_x = np.exp(x)
    return exp_x/np.sum(exp_x)

In [ ]:
# ニューラルネットワークにおける勾配の計算
class simpleNet:
    def __init__(self):
        # ガウス分布で初期化
        self.W = np.random.randn(2, 3)

    def predict(self, x):
        # 入力xと重み行列Wのドット積を計算して、中間層の出力zを得る
        z = np.dot(x, self.W)
        # softmax関数を適用して、出力層の出力を得る
        return softmax(z)

    def loss(self, x, t):
        # 入力xと重み行列Wのドット積を計算して、中間層の出力zを得る
        z = np.dot(x, self.W)
        # softmax関数を適用して、出力層の出力yを得る
        y = softmax(z)
        # 交差エントロピー誤差関数を適用して、損失を計算する
        loss = cross_entropy_error(y, t)

        return loss   # 損失を返す

In [ ]:
# モデルの作成
network = simpleNet()

# 重みの確認
print(network.W)

In [ ]:
# データから予測する
x = np.array([0.6, 0.9])
y = network.predict(x)
print(y)

In [ ]:
# 最大のインデックスを表示する
np.argmax(y)

In [ ]:
# 教師データは「2」を正解とする
t = np.array([0, 0, 1])
network.loss(x, t)

In [ ]:
# 損失関数を用いて重みを更新する
def loss_f(W):
    return network.loss(x, t)

In [ ]:
dW = numerical_gradient(loss_f, network.W)
print(dW)

In [ ]:
# 損失から重みを更新する
print('更新前の重み：', network.W)

# 学習率
lr = 0.01
network.W -= lr * dW
print('\n更新後の重み：', network.W)